<a href="https://colab.research.google.com/github/gramucse/ai-mentor-portfolio/blob/main/Day5_HF_Pulls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5 Lab 5B — Hugging Face Pulls (API + local)**Goal:** Same model two ways. Build the timing table.**Time:** 90 minutes.

## Step 1 — Install + token

In [2]:
!pip install -q transformers requests sentence-transformers

import os
import getpass

if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass.getpass('HF token (read scope): ')

HF token (read scope): ··········


## Step 2 — Zero-shot via Inference API

In [6]:
import os, requests, time

HF_TOKEN = os.environ["HF_TOKEN"]

def hf_zero_shot_api(text, labels):
    url = "https://api-inference.huggingface.co/models/facebook/bart-large-mnli"

    r = requests.post(
        url,
        headers={"Authorization": f"Bearer {HF_TOKEN}"},
        json={
            "inputs": text,
            "parameters": {"candidate_labels": labels}
        }
    )

    print("Status code:", r.status_code)

    try:
        return r.json()
    except Exception:
        print("Raw response:")
        print(r.text)
        return {"error": "Response was not valid JSON"}

resumes = [
    "Built React dashboards for 3 startups",
    "Implemented Spring Boot microservices in Java for fintech app",
    "Trained CNN for image classification using PyTorch, 87% accuracy",
    "Cleaned 100k row dataset using pandas + plotted in seaborn for monthly reports",
    "Wrote SQL queries against PostgreSQL, optimised 3 slow queries by 10x",
]

labels = ["frontend dev", "backend dev", "data analyst", "ML engineer"]

start = time.time()

for r in resumes:
    result = hf_zero_shot_api(r, labels)

    if "error" in result:
        print("Error:", result["error"])
    else:
        print(f'{r[:50]:50} -> {result["labels"][0]} ({result["scores"][0]:.2f})')

print(f"\nAPI time: {time.time() - start:.2f}s")

Status code: 404
Raw response:
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot POST /models/facebook/bart-large-mnli</pre>
</body>
</html>

Error: Response was not valid JSON
Status code: 404
Raw response:
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot POST /models/facebook/bart-large-mnli</pre>
</body>
</html>

Error: Response was not valid JSON
Status code: 404
Raw response:
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot POST /models/facebook/bart-large-mnli</pre>
</body>
</html>

Error: Response was not valid JSON
Status code: 404
Raw response:
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot POST /models/facebook/bart-large-mnli</pre>
</body>
</html>

Error: Response was not valid JSON
Status code: 404
Raw response:
<!DOCTYPE html>
<

## Step 3 — Same model locally

In [7]:
from transformers import pipeline

# This DOWNLOADS the model on first run — ~1.6GB. Be patient.
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

start = time.time()
for r in resumes:
    res = classifier(r, candidate_labels=labels)
    print(f'  {r[:50]:50} -> {res["labels"][0]} ({res["scores"][0]:.2f})')
print(f'\nLocal time (after download): {time.time()-start:.2f}s')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Built React dashboards for 3 startups              -> frontend dev (0.88)
  Implemented Spring Boot microservices in Java for  -> backend dev (0.63)
  Trained CNN for image classification using PyTorch -> ML engineer (0.40)
  Cleaned 100k row dataset using pandas + plotted in -> data analyst (0.56)
  Wrote SQL queries against PostgreSQL, optimised 3  -> data analyst (0.45)

Local time (after download): 11.73s


## Step 4 — Sentiment analysis

In [8]:
sentiment = pipeline('sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english')

# Test data — 5 mock interview answers
answers = [
    'I really enjoyed working on the team and shipped 3 features.',
    'I was the only one writing code; everyone else was slow.',
    'I learned a lot from my mentor and grew technically.',
    "I had to redo most of my teammate's work because it was wrong.",
    'My internship was great — would recommend it to anyone.',
]

print('Sentiment scores:')
for a in answers:
    result = sentiment(a)[0]
    label = result['label']
    score = result['score']
    print(f'  [{label} {score:.2f}] {a[:60]}')

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Sentiment scores:
  [POSITIVE 1.00] I really enjoyed working on the team and shipped 3 features.
  [NEGATIVE 1.00] I was the only one writing code; everyone else was slow.
  [POSITIVE 1.00] I learned a lot from my mentor and grew technically.
  [NEGATIVE 1.00] I had to redo most of my teammate's work because it was wron
  [POSITIVE 1.00] My internship was great — would recommend it to anyone.


In [9]:
import time

def time_call(fn, n_runs=3):
    times = []
    for _ in range(n_runs):
        start = time.time()
        fn()
        times.append(time.time() - start)
    return min(times), sum(times)/len(times)

# Time API call (warm)
def call_api():
    hf_zero_shot_api('Built React dashboards', ['frontend dev', 'backend dev'])
api_min, api_avg = time_call(call_api)

# Time local call (warm)
def call_local():
    classifier('Built React dashboards', candidate_labels=['frontend dev', 'backend dev'])
local_min, local_avg = time_call(call_local)

print(f'Inference timing comparison (3 runs each, after warm-up):')
print(f'  API:   min {api_min:.2f}s | avg {api_avg:.2f}s')
print(f'  Local: min {local_min:.2f}s | avg {local_avg:.2f}s')

Status code: 404
Raw response:
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot POST /models/facebook/bart-large-mnli</pre>
</body>
</html>

Status code: 404
Raw response:
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot POST /models/facebook/bart-large-mnli</pre>
</body>
</html>

Status code: 404
Raw response:
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot POST /models/facebook/bart-large-mnli</pre>
</body>
</html>

Inference timing comparison (3 runs each, after warm-up):
  API:   min 0.07s | avg 0.09s
  Local: min 0.66s | avg 0.74s


## Step 5 — Timing tableWrite the API vs local timing comparison in your README + 3-line reflection on when to use each in production.

Inference timing comparison (3 runs each, after warm-up):
  API:   min 0.07s | avg 0.09s
  Local: min 0.66s | avg 0.74s